In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [3]:
class BasicExpert(nn.Module):
    def __init__(self, feature_in, feature_out):
        super().__init__()
        self.linear = nn.Linear(feature_in, feature_out)

    def forward(self, X):
        return self.linear(X)
    

In [4]:
class BasicMOE(nn.Module):
    def __init__(self, feature_in, feature_out, num_experts):
        super().__init__()
        self.experts = nn.ModuleList(
            [
                BasicExpert(feature_in, feature_out) for _ in range(num_experts)
            ]
        )
        self.gate = nn.Linear(feature_in, num_experts)
    
    def forward(self, X):
        expert_weight = self.gate(X)   #(batch, num_experts)
        expert_out_list = [
            expert(X).unsqueeze(1) for expert in self.experts
        ]
        expert_output = torch.concat(expert_out_list, dim=1)

        expert_weight = F.softmax(expert_weight, dim=-1).unsqueeze(1)
        output = expert_weight @ expert_output

        return output.squeeze()

In [5]:
X = torch.rand(2, 5)
basic_moe = BasicMOE(5, 8, 4)
out = basic_moe(X)
print(out)

tensor([[ 0.0010,  0.2966,  0.1273,  0.0704,  0.0122,  0.0193, -0.0977,  0.1347],
        [-0.1224,  0.4667,  0.1014,  0.1411,  0.0156,  0.1108, -0.1601,  0.1197]],
       grad_fn=<SqueezeBackward0>)


In [10]:
import torch
import torch.nn.functional as F


# 创建一个 3D 张量，方便观察
# shape: (2, 3, 4) = (batch=2, seq_len=3, expert_num=4)
x = torch.tensor([
    # batch 0
    [
        [1.0, 5.0, 3.0, 2.0],   # token 0 对4个专家的分数
        [4.0, 2.0, 6.0, 1.0],   # token 1 对4个专家的分数
        [3.0, 7.0, 2.0, 8.0],   # token 2 对4个专家的分数
    ],
    # batch 1
    [
        [2.0, 1.0, 9.0, 3.0],   # token 0
        [5.0, 4.0, 3.0, 2.0],   # token 1
        [6.0, 1.0, 7.0, 8.0],   # token 2
    ]
])

x = F.softmax(x, dim=-1)
print(f"原始张量 shape: {x.shape}")
print(x)
values, indices = torch.topk(x, k=2, dim=-1)  # 或 dim=2

print(f"values shape:  {values.shape}")   # (2, 3, 2)
print(f"indices shape: {indices.shape}")  # (2, 3, 2)

原始张量 shape: torch.Size([2, 3, 4])
tensor([[[1.5219e-02, 8.3095e-01, 1.1246e-01, 4.1371e-02],
         [1.1663e-01, 1.5784e-02, 8.6178e-01, 5.8066e-03],
         [4.8929e-03, 2.6714e-01, 1.8000e-03, 7.2617e-01]],

        [[9.0850e-04, 3.3422e-04, 9.9629e-01, 2.4696e-03],
         [6.4391e-01, 2.3688e-01, 8.7144e-02, 3.2059e-02],
         [8.9976e-02, 6.0625e-04, 2.4458e-01, 6.6484e-01]]])
values shape:  torch.Size([2, 3, 2])
indices shape: torch.Size([2, 3, 2])


In [11]:
print(values)

tensor([[[0.8310, 0.1125],
         [0.8618, 0.1166],
         [0.7262, 0.2671]],

        [[0.9963, 0.0025],
         [0.6439, 0.2369],
         [0.6648, 0.2446]]])


In [12]:
print(indices)

tensor([[[1, 2],
         [2, 0],
         [3, 1]],

        [[2, 3],
         [0, 1],
         [3, 2]]])
